# 第 3 週：訓練關鍵點模型（U-Net + ResNet18）

**使用方式：** 由上到下，一格一格按左邊的 ▶ 執行。

開始前先確認兩件事：
1. 上方選單 **執行階段 → 變更執行階段類型 → 選 T4 GPU → 儲存**。
2. Google Drive 的 `我的雲端硬碟/EchoLVH/` 裡面有資料 zip（第二版：`echolvh_256_full.zip`）。

模型會自動存到 `我的雲端硬碟/EchoLVH/checkpoints/<RUN_NAME>/`（第二版是 `v2_full`，不會蓋掉第一版）。中途斷線也不怕，重新從頭執行會接著訓練。

## 1. 確認有 GPU
看到 `Tesla T4` 就對了；如果出現錯誤，回去做上面第 1 點。

In [ ]:
!nvidia-smi

## 2. 連接 Google Drive、解壓資料
會跳出授權視窗，選你的 Google 帳號並允許。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/EchoLVH'
ZIP_NAME = 'echolvh_256_full.zip'   # 第一版是 echolvh_256.zip（2,400 張）；第二版用全部訓練資料
RUN_NAME = 'v2_full'                # 這次的模型存在 checkpoints/v2_full/
CKPT_DIR = f'{DRIVE_DIR}/checkpoints/{RUN_NAME}'
!mkdir -p "$CKPT_DIR"
!rm -rf /content/echolvh_256 && cp "$DRIVE_DIR/$ZIP_NAME" /content/data.zip && cd /content && unzip -q -o data.zip
!ls /content/echolvh_256 | head; ls /content/echolvh_256/images | wc -l

## 3. 安裝套件

In [ ]:
!pip install -q segmentation-models-pytorch

## 4. 設定
想調整訓練時，大多只要改這一格：
- `EPOCHS`：訓練幾輪。第二版用 25 輪，大約 1 小時（第一版 2,400 張時用 40 輪）。
- `SIGMA`：熱力圖光點的大小（像素）。

In [ ]:
import os, json, math, random, time
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn
import matplotlib.pyplot as plt

DATA_DIR = '/content/echolvh_256'
KP = ['lvpw_outer', 'lvpw_lvid', 'lvid_ivs', 'ivs_outer']   # 由下往上 4 個點
SEGS = [('LVPWd', 0, 1), ('LVIDd', 1, 2), ('IVSd', 2, 3)]    # 量測 = 兩點距離
IMG = 256
SIGMA = 2.0
BATCH = 16
EPOCHS = 25   # 資料是第一版的 4 倍，每輪也變 4 倍久；25 輪約 1 小時
LR = 1e-3
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
labels = pd.read_csv(f'{DATA_DIR}/labels.csv')
ok = np.array([cv2.imread(f'{DATA_DIR}/images/{i}.png', cv2.IMREAD_GRAYSCALE) is not None for i in labels.id])
if not ok.all():
    print('⚠ 略過讀不到的圖片：', labels.id[~ok].tolist())
    labels = labels[ok].reset_index(drop=True)
print(labels.split.value_counts().to_dict(), '| device:', DEVICE)

## 5. 資料集
- 每張圖 → 4 張熱力圖（每個點一張，正確位置畫一個高斯光點）。
- 訓練時做**資料增強**：小角度旋轉、縮放、平移、亮度對比變化（點的座標同步轉換）。
  不做左右翻轉，因為心臟方向是固定的。

In [ ]:
from torch.utils.data import Dataset, DataLoader

def make_heatmaps(pts, size=IMG, sigma=SIGMA):
    ys, xs = np.mgrid[0:size, 0:size].astype(np.float32)
    return np.stack([np.exp(-((xs - x) ** 2 + (ys - y) ** 2) / (2 * sigma ** 2)) for x, y in pts]).astype(np.float32)

class EchoKP(Dataset):
    def __init__(self, df, train):
        self.df, self.train = df.reset_index(drop=True), train
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = cv2.imread(f'{DATA_DIR}/images/{r.id}.png', cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        pts = np.array([[r[f'{k}_x'], r[f'{k}_y']] for k in KP], dtype=np.float32)
        if self.train:
            ang = np.random.uniform(-10, 10); sc = np.random.uniform(0.9, 1.1)
            tx, ty = np.random.uniform(-12, 12, size=2)
            M = cv2.getRotationMatrix2D((IMG / 2, IMG / 2), ang, sc); M[:, 2] += (tx, ty)
            img = cv2.warpAffine(img, M, (IMG, IMG), flags=cv2.INTER_LINEAR, borderValue=0)
            pts = (np.c_[pts, np.ones(4)] @ M.T).astype(np.float32)
            img = np.clip(img * np.random.uniform(0.8, 1.2) + np.random.uniform(-0.08, 0.08), 0, 1)
        x = torch.from_numpy(np.repeat(img[None], 3, axis=0))          # 灰階複製成 3 通道（配合預訓練權重）
        x = (x - 0.45) / 0.225
        return x, torch.from_numpy(make_heatmaps(pts)), torch.from_numpy(pts), torch.tensor(r.cm_per_px_256, dtype=torch.float32)

dl = {s: DataLoader(EchoKP(labels[labels.split == s], s == 'train'), batch_size=BATCH,
                    shuffle=(s == 'train'), num_workers=2, drop_last=(s == 'train'))
      for s in ['train', 'val', 'test']}

# 看一張增強後的訓練圖，確認點還在心壁上
x, hm, pts, _ = dl['train'].dataset[0]
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1); plt.imshow(x[0], cmap='gray'); plt.scatter(pts[:, 0], pts[:, 1], c='r', s=12); plt.title('Augmented image + keypoints')
plt.subplot(1, 2, 2); plt.imshow(hm.max(0).values, cmap='hot'); plt.title('4 heatmaps (max)'); plt.show()

## 6. 模型、損失函數、評估方式
- **模型**：`smp.Unet`，編碼器是 ImageNet 預訓練的 ResNet18，輸出 4 個通道。
- **損失**：空間 softmax + 交叉熵。把每張熱力圖當成「在 256×256 個像素中選出正確位置」的分類問題，目標是以正確點為中心的高斯分布。比 MSE 穩定，不會學成全部輸出 0。
- **解碼**：softmax 後取機率最高的點，再用周圍像素加權平均做到次像素精度。
- **評估**：把預測的點換算成 IVSd / LVIDd / LVPWd，和醫師的值比較，單位 mm。

In [ ]:
import segmentation_models_pytorch as smp

def build_model(pretrained=True):
    return smp.Unet('resnet18', encoder_weights='imagenet' if pretrained else None, in_channels=3, classes=4)

def to_prob(logits):
    B, K, H, W = logits.shape
    return logits.flatten(2).softmax(-1).view(B, K, H, W)

def heatmap_loss(logits, gt):
    # 空間 softmax 交叉熵：目標 = 高斯熱力圖正規化成機率分布
    logp = logits.flatten(2).log_softmax(-1)
    target = gt.flatten(2); target = target / target.sum(-1, keepdim=True).clamp(min=1e-6)
    return -(target * logp).sum(-1).mean()

def decode(hm, win=2):
    # hm: (B,4,H,W) → (B,4,2) 座標；argmax + 周圍 (2*win+1)^2 加權平均
    B, K, H, W = hm.shape
    idx = hm.flatten(2).argmax(-1)
    cy, cx = (idx // W).float(), (idx % W).float()
    ys = torch.arange(H, device=hm.device).float(); xs = torch.arange(W, device=hm.device).float()
    my = ((ys.view(1, 1, H) - cy[..., None]).abs() <= win).float()
    mx = ((xs.view(1, 1, W) - cx[..., None]).abs() <= win).float()
    w = hm.clamp(min=0) * my[..., :, None] * mx[..., None, :]
    s = w.sum((-1, -2)).clamp(min=1e-6)
    return torch.stack([(w.sum(-2) * xs).sum(-1) / s, (w.sum(-1) * ys).sum(-1) / s], -1)

def measures(pts, cmpp):
    # pts (B,4,2), cmpp (B,) → (B,3) 單位 cm，順序 LVPWd, LVIDd, IVSd
    return torch.stack([(pts[:, b] - pts[:, a]).norm(dim=-1) * cmpp for _, a, b in SEGS], -1)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, T, PX = [], [], []
    for x, hm, pts, cmpp in loader:
        pred = decode(to_prob(model(x.to(DEVICE)).float())).cpu()
        P.append(measures(pred, cmpp)); T.append(measures(pts, cmpp)); PX.append((pred - pts).norm(dim=-1))
    P, T, PX = torch.cat(P), torch.cat(T), torch.cat(PX)
    mae = ((P - T).abs() * 10).mean(0)   # mm
    return {'mae_mm': dict(zip([s[0] for s in SEGS], mae.tolist())), 'mean_mae_mm': mae.mean().item(),
            'point_err_px': PX.mean().item(), 'pred': P.numpy(), 'true': T.numpy()}

## 7. 訓練
每一輪會印出 train loss 與 val 的平均誤差（mm）。val 誤差最好的模型會存到 Drive 的 `best.pt`。
如果中途斷線：重新從第 1 格開始執行，這格會從 `last.pt` 接著訓練。

In [ ]:
model = build_model().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl['train']), pct_start=0.1)
scaler = torch.amp.GradScaler(enabled=(DEVICE == 'cuda'))
start, best, history = 0, float('inf'), []

if os.path.exists(f'{CKPT_DIR}/last.pt'):
    ck = torch.load(f'{CKPT_DIR}/last.pt', map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
    start, best, history = ck['epoch'] + 1, ck['best'], ck['history']
    print(f'從第 {start} 輪接著訓練（目前最佳 {best:.2f} mm）')

for ep in range(start, EPOCHS):
    model.train(); t0 = time.time(); tot = 0
    for x, hm, _, _ in dl['train']:
        x, hm = x.to(DEVICE), hm.to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(DEVICE == 'cuda')):
            loss = heatmap_loss(model(x).float(), hm)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item()
    v = evaluate(model, dl['val'])
    history.append({'epoch': ep, 'loss': tot / len(dl['train']), 'val_mae_mm': v['mean_mae_mm'], **v['mae_mm']})
    flag = ''
    if v['mean_mae_mm'] < best:
        best = v['mean_mae_mm']; flag = '  ← 最佳，已存 best.pt'
        torch.save({'model': model.state_dict(), 'epoch': ep, 'val': v['mae_mm'],
                    'config': {'arch': 'Unet', 'encoder': 'resnet18', 'img': IMG, 'keypoints': KP}}, f'{CKPT_DIR}/best.pt')
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                'epoch': ep, 'best': best, 'history': history}, f'{CKPT_DIR}/last.pt')
    m = v['mae_mm']
    print(f"第 {ep+1:2d}/{EPOCHS} 輪 | loss {tot/len(dl['train']):.5f} | val 誤差 IVSd {m['IVSd']:.2f} LVIDd {m['LVIDd']:.2f} "
          f"LVPWd {m['LVPWd']:.2f} mm | 點誤差 {v['point_err_px']:.1f}px | {time.time()-t0:.0f}s{flag}")

## 8. 訓練曲線

In [ ]:
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h.epoch + 1, h.loss); ax[0].set_title('Train loss'); ax[0].set_xlabel('epoch')
for c in ['IVSd', 'LVIDd', 'LVPWd']: ax[1].plot(h.epoch + 1, h[c], label=c)
ax[1].set_title('Val MAE (mm)'); ax[1].set_xlabel('epoch'); ax[1].legend(); plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/curves.png', dpi=120); plt.show()

## 9. 最終測試（test set，訓練時完全沒看過）
報告要用的數字都在這裡：每個量測的 MAE（mm）、R²、肥厚判斷準確率，加上「預測 vs 真實」散佈圖。

In [ ]:
ck = torch.load(f'{CKPT_DIR}/best.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(ck['model'])
t = evaluate(model, dl['test'])
P, T = t['pred'], t['true']
names = [s[0] for s in SEGS]
r2 = {n: 1 - ((P[:, i] - T[:, i]) ** 2).sum() / ((T[:, i] - T[:, i].mean()) ** 2).sum() for i, n in enumerate(names)}
lvh_t = np.maximum(T[:, 0], T[:, 2]) >= 1.1; lvh_p = np.maximum(P[:, 0], P[:, 2]) >= 1.1
acc = (lvh_t == lvh_p).mean(); sens = (lvh_p & lvh_t).sum() / max(lvh_t.sum(), 1); spec = (~lvh_p & ~lvh_t).sum() / max((~lvh_t).sum(), 1)

result = {'mae_mm': t['mae_mm'], 'r2': {k: float(v) for k, v in r2.items()}, 'point_err_px': t['point_err_px'],
          'lvh_accuracy': float(acc), 'lvh_sensitivity': float(sens), 'lvh_specificity': float(spec),
          'n_test': int(len(P)), 'best_epoch': int(ck['epoch']) + 1}
json.dump(result, open(f'{CKPT_DIR}/test_result.json', 'w'), indent=2, ensure_ascii=False)
print(json.dumps(result, indent=2, ensure_ascii=False))

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for i, n in enumerate(names):
    ax[i].scatter(T[:, i], P[:, i], s=8, alpha=0.6)
    lo, hi = T[:, i].min(), T[:, i].max(); ax[i].plot([lo, hi], [lo, hi], 'r--', lw=1)
    ax[i].set_title(f"{n}  MAE {t['mae_mm'][n]:.2f} mm  R² {r2[n]:.2f}"); ax[i].set_xlabel('Doctor (cm)'); ax[i].set_ylabel('Model (cm)')
plt.tight_layout(); plt.savefig(f'{CKPT_DIR}/test_scatter.png', dpi=120); plt.show()

## 10. 看看模型實際標的點（綠 = 醫師，紅 = 模型）

In [ ]:
model.eval(); ds = dl['test'].dataset
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for a, i in zip(ax.flat, np.random.RandomState(0).choice(len(ds), 8, replace=False)):
    x, _, pts, cmpp = ds[i]
    with torch.no_grad(): pred = decode(to_prob(model(x[None].to(DEVICE)).float())).cpu()[0]
    m_t, m_p = measures(pts[None], cmpp[None])[0], measures(pred[None], cmpp[None])[0]
    a.imshow(x[0], cmap='gray'); a.plot(pts[:, 0], pts[:, 1], 'g.-'); a.plot(pred[:, 0], pred[:, 1], 'r.-')
    a.set_title(f'IVSd {m_t[2]:.2f}/{m_p[2]:.2f}  LVPWd {m_t[0]:.2f}/{m_p[0]:.2f}', fontsize=9); a.axis('off')
plt.suptitle('doctor / model (cm)   green = doctor, red = model'); plt.tight_layout(); plt.savefig(f'{CKPT_DIR}/test_examples.png', dpi=110); plt.show()

## 完成！
Drive 的 `EchoLVH/checkpoints/v2_full/` 裡現在有：
- `best.pt`：訓練好的模型，第 4 週後端會用到
- `test_result.json`、`test_scatter.png`、`curves.png`、`test_examples.png`：報告要用的結果

把 `test_result.json` 的內容貼給 Claude，一起看下一步怎麼改進。